# Volatility targeting con bandas

> ¿Pueden las bandas de no negociación reducir los costos sin deteriorar de forma importante el control del riesgo?

Trabajaremos por bloques. Hoy sólo hacemos el **Bloque 01**: confirmar que SPY está disponible y que sus precios sirven para el experimento.

## Bloque 01 — Datos de SPY

Usamos la serie diaria **Gold** aprobada por el quality gate. Revisamos cuatro cosas sencillas:

1. Cobertura temporal.
2. Fechas únicas y ordenadas.
3. Precios de apertura, máximo, mínimo, cierre y cierre ajustado.
4. Dividendos y splits, para saber qué ajustes existen.

In [ ]:
from pathlib import Path

import polars as pl

def find_project_root() -> Path:
    # Funciona si Jupyter abre la notebook desde volatility_target o desde su carpeta padre.
    for folder in (Path.cwd(), *Path.cwd().parents):
        for candidate in (folder, folder / 'volatility_target'):
            if (candidate / 'data' / 'gold' / 'daily_series' / 'etf' / 'SPY.parquet').exists():
                return candidate
    raise FileNotFoundError('No encontré data/gold/daily_series/etf/SPY.parquet. Ejecuta primero el pipeline de SPY.')


project_root = find_project_root()
spy_path = project_root / 'data' / 'gold' / 'daily_series' / 'etf' / 'SPY.parquet'
spy = pl.read_parquet(spy_path)

print(f"Filas: {spy.height:,}")
print(f"Periodo: {spy['session_date'].min()} a {spy['session_date'].max()}")

spy.select([
    'session_date', 'open', 'close', 'adjusted_close',
    'dividend_amount', 'split_coefficient',
]).tail(5)

In [ ]:
price_columns = ['open', 'high', 'low', 'close', 'adjusted_close']

checks = {
    'Fechas únicas': spy['session_date'].n_unique() == spy.height,
    'Fechas ordenadas': spy['session_date'].to_list() == sorted(spy['session_date'].to_list()),
    'Sin precios faltantes': sum(spy[column].null_count() for column in price_columns) == 0,
    'OHLC coherente': spy.select(
        ((pl.col('high') >= pl.max_horizontal('open', 'close', 'low')) &
         (pl.col('low') <= pl.min_horizontal('open', 'close', 'high'))).all()
    ).item(),
}

for name, passed in checks.items():
    print(f"{'✓' if passed else '✗'} {name}")

print(f"Sesiones con dividendo: {spy.filter(pl.col('dividend_amount') > 0).height}")
print(f"Sesiones con split: {spy.filter(pl.col('split_coefficient') != 1).height}")

### Conclusión del Bloque 01

La serie Gold de SPY cubre del **2018-01-02** al **2026-11-10** y contiene **2,227** sesiones. La auditoría aprobó la serie: no hay fechas duplicadas, precios faltantes ni inconsistencias OHLC. Hay 35 sesiones con dividendos y ningún split en este periodo.

Por tanto, tenemos precios de apertura, cierre y cierre ajustado suficientes para continuar después con EWMA, sin implementar todavía ninguna política de rebalanceo.